# Protein Aggregation and Structural Measurements under Macromolecular Crowding in Escherichia coli Cell-Free Translation System Exploration with `mlcroissant`
This notebook guides you through loading and exploring the FAIR^2 dataset using the `mlcroissant` library, following FAIR principles and leveraging Croissant schema definitions.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

```
https://sen.science/doi/10.71728/senscience.yh1d-960v/fair2.json
```

The metadata provides descriptions, fields, and record sets for protein aggregation and structure measurements in Escherichia coli.

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.yh1d-960v/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Display dataset metadata using attributes
print('Dataset Name:', dataset.metadata.name)
print('Description:', dataset.metadata.description)
print('Published Date:', dataset.metadata.datePublished)
print('Keywords:', ', '.join(dataset.metadata.keywords))

## 2. Data Overview
Review available record sets, fields, and their IDs.

Record sets, fields, and columns are referenced by their `@id`.

In [ ]:
# List all available record sets in the dataset
record_sets = dataset.metadata.recordSet

if not record_sets:
    print("No record sets found in this dataset's metadata.")
else:
    print('Record sets found:')
    for rs in record_sets:
        print(f"- @id: {rs['@id']}, name: {rs.get('name', 'N/A')}")

# Show fields for each record set, referenced by @id
for idx, rs in enumerate(record_sets):
    print(f"\nRecord Set #{idx+1} @id: {rs['@id']}")
    fields = rs.get('field', [])
    for field in fields:
        print(f"  Field @id: {field['@id']}, name: {field.get('name', 'N/A')}, type: {field.get('dataType', 'N/A')}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

Below, we extract the data from all available record sets, referencing each by its `@id`.

In [ ]:
# Prepare to load all available record sets
dataframes = {}
record_set_ids = []

# Collect record set @ids
if dataset.metadata.recordSet:
    record_set_ids = [rs['@id'] for rs in dataset.metadata.recordSet]

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df

# Show the columns for each record set
for record_set_id in dataframes:
    print(f"Record Set @id: {record_set_id}")
    print('Columns:', dataframes[record_set_id].columns.tolist())
    print(dataframes[record_set_id].head(), '\n')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. Use `@id` references for record sets and fields throughout.

Below, we select a numeric field, filter by value, normalize it, and optionally group by another categorical field.

In [ ]:
# Choose the first record set for demonstration
if record_set_ids:
    selected_record_set_id = record_set_ids[0]
    df = dataframes[selected_record_set_id]
else:
    selected_record_set_id = None
    df = None

if df is not None:
    # Inspect column names for candidates
    print("Columns available:", df.columns.tolist())
    # Try to select a numeric field e.g. 'molecular_weight' or similar
    # We'll assume some plausible column names relevant to protein measurements
    numeric_field_candidates = [col for col in df.columns if col.lower() in ['molecular_weight', 'yield', 'aggregation_propensity', 'radius_gyration', 'surface_area'] or df[col].dtype in ['float64', 'int64']]
    
    if numeric_field_candidates:
        numeric_field = numeric_field_candidates[0]
        print(f"Using numeric field '{numeric_field}' for analysis.")
    else:
        numeric_field = df.columns[0]  # fallback
        print(f"No clear numeric field found, defaulting to '{numeric_field}'.")

    # Filter where value > threshold
    threshold = df[numeric_field].median() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 10
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold}:")
    print(filtered_df.head())

    # Normalize the numeric field
    if pd.api.types.is_numeric_dtype(filtered_df[numeric_field]):
        filtered_df[numeric_field + '_normalized'] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, numeric_field + '_normalized']].head())
    else:
        print(f"Field '{numeric_field}' is not numeric, skipping normalization.")

    # Try grouping by a categorical field (e.g. 'SCOP_superfamily', 'protein_name')
    group_field_candidates = [col for col in df.columns if col.lower() in ['scop_superfamily', 'protein_name', 'category', 'name'] or df[col].dtype == 'object']
    group_field = None
    for col in group_field_candidates:
        if col != numeric_field:
            group_field = col
            break
    
    if group_field and group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"Grouped data by '{group_field}':")
        print(grouped_df.head())
    else:
        print("No suitable group field found for grouping.")

## 5. Visualization
Visualize data distributions or relationships between fields using matplotlib or seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and numeric_field is not None:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field].dropna(), bins=15, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    if group_field and group_field in df.columns and pd.api.types.is_numeric_dtype(df[numeric_field]):
        # Boxplot of numeric field per group
        plt.figure(figsize=(10, 6))
        sns.boxplot(x=df[group_field], y=df[numeric_field])
        plt.xticks(rotation=45)
        plt.title(f"{numeric_field} by {group_field}")
        plt.show()

## 6. Conclusion
This notebook demonstrated how to load, extract, and analyze a Croissant-structured FAIR^2 dataset of E. coli protein measurements using `mlcroissant`. By referencing fields and record sets by their `@id`, users can ensure reproducible, schema-driven data analysis workflows.

- Dataset metadata and structure were explored directly from the schema.
- Tabular protein measurement records were loaded by record set `@id`.
- Common analysis operations were performed, including filtering, normalization, grouping, and visualizations.

The FAIR^2 dataset supports advanced analysis and modeling of protein aggregation and structural properties in macromolecular crowding contexts.

<!-- End of notebook -->